# Attention Sinks Experiment — Gemma 4 E2B

**Question.** Gemma 4 E2B uses sliding-window attention (window = 512) on 28 of its 35 layers.
Once a query is past position 512, the BOS token is evicted. Where does that attention mass go,
and does it still pull on the residual stream?

**What Cancedda 2024 claims.** BOS is a cost-free dump — its value vector has low norm,
so attention mass that lands there produces no residual change.

**How we falsify it.** We compute two scores for every key position k at late queries q ≥ 512:

- `score_pre(q, k)` — L2 norm of k's contribution vector to the pre-norm attention output:
  `‖Σ_h a[q,k,h] · (V[h,k] @ W_o[h]ᵀ)‖₂`

- `score_exact(q, k)` — leave-one-out shift in post-RMSNorm space:
  `‖RMSNorm(pre_out) − RMSNorm(pre_out − c_k)‖₂`

If `score_exact(BOS) > 0` in full layers, BOS is not a no-op.

**Position groups.**
`bos` · `edge` (first 4 visible) · `self` · `recent_nonself` (last 32 excl. self) · `middle`


In [1]:
import json, time, types
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoProcessor
from transformers.models.gemma4 import modeling_gemma4

# ── edit these two lines for smoke vs full run ───────────────────────────
TARGET_PER_STRATUM = 1        # smoke = 1  │  full run = 20
SAMPLING_MODE      = "first"  # smoke = "first"  │  full run = "random"
# ─────────────────────────────────────────────────────────────────────────

MODEL_ID    = "google/gemma-4-E2B-it"
DATA_DIR    = Path("data/TowerBlocks-v0.1")
DEVICE      = "mps"
SAMPLE_SEED = 0
MAX_LENGTH  = 640     # truncation cap; must exceed 512 to evict BOS
ANCHOR_K    = 8       # top-k positions scored exactly at anchor queries

OUT_DIR = Path("outputs/smoke") if TARGET_PER_STRATUM < 5 else Path("outputs/full")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"run: {'smoke' if TARGET_PER_STRATUM < 5 else 'full'}   outputs → {OUT_DIR}/")

STRATA = [
    dict(name="chat/en",                              task="chat",                            lang="en"),
    dict(name="named_entity_recognition/en",          task="named_entity_recognition",        lang="en"),
    dict(name="named_entity_recognition/es",          task="named_entity_recognition",        lang="es"),
    dict(name="machine_translation/en-de",            task="machine_translation",             lang="en-de"),
    dict(name="machine_translation_evaluation/zh_en", task="machine_translation_evaluation",  lang="zh_en"),
]


run: smoke   outputs → outputs/smoke/


## 1. Load model

In [2]:
processor     = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer     = processor.tokenizer
cfg           = AutoConfig.from_pretrained(MODEL_ID).text_config
layer_types   = list(cfg.layer_types)
sliding_window = int(cfg.sliding_window)

print(f"{len(layer_types)} layers  ·  {layer_types.count('sliding_attention')} sliding "
      f"(window={sliding_window})  ·  {layer_types.count('full_attention')} full")
print(f"BOS evicted from sliding layers when q ≥ {sliding_window}\n")
for i, t in enumerate(layer_types):
    print(f"  layer {i:2d}  {'S' if 'sliding' in t else 'F'}")


chat_template.jinja: 0.00B [00:00, ?B/s]

35 layers  ·  28 sliding (window=512)  ·  7 full
BOS evicted from sliding layers when q ≥ 512

  layer  0  S
  layer  1  S
  layer  2  S
  layer  3  S
  layer  4  F
  layer  5  S
  layer  6  S
  layer  7  S
  layer  8  S
  layer  9  F
  layer 10  S
  layer 11  S
  layer 12  S
  layer 13  S
  layer 14  F
  layer 15  S
  layer 16  S
  layer 17  S
  layer 18  S
  layer 19  F
  layer 20  S
  layer 21  S
  layer 22  S
  layer 23  S
  layer 24  F
  layer 25  S
  layer 26  S
  layer 27  S
  layer 28  S
  layer 29  F
  layer 30  S
  layer 31  S
  layer 32  S
  layer 33  S
  layer 34  F


In [3]:
print("loading weights …")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, attn_implementation="eager", dtype=torch.bfloat16)
model.eval().to(DEVICE)
print(f"ready in {time.time()-t0:.0f}s")


loading weights …


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

ready in 58s


## 2. Sample prompts

We pull from TowerBlocks (a stratified multilingual instruction dataset),
keep only prompts that are long enough for BOS eviction (`seq_len > 512`),
and truncate to `MAX_LENGTH`.

Sampling modes:
- `"first"` — deterministic, takes the first eligible prompt per stratum (fast smoke test)
- `"random"` — reservoir sampling (representative full run)


In [4]:
import random

def tokenise(msgs, max_len):
    ids = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=False,
        tokenize=True, return_dict=True, return_tensors="pt"
    )["input_ids"]
    return ids[0, :max_len].tolist()

def parse_conversations(row):
    role_map = {"user":"user","human":"user","assistant":"assistant","gpt":"assistant","system":"system"}
    msgs = []
    for turn in (row.get("conversations") or []):
        role    = role_map.get(turn.get("from") or turn.get("role"))
        content = turn.get("value") or turn.get("content")
        if role and isinstance(content, str) and content.strip():
            msgs.append({"role": role, "content": content})
    has_user = any(m["role"] == "user"      for m in msgs)
    has_asst = any(m["role"] == "assistant" for m in msgs)
    return msgs if (has_user and has_asst) else None

rng          = random.Random(SAMPLE_SEED)
stratum_idx  = {(s["task"], s["lang"]): s for s in STRATA}
reservoirs   = {s["name"]: [] for s in STRATA}
eligible     = {s["name"]: 0  for s in STRATA}
scanned = skipped_parse = skipped_short = 0

parquet_files = sorted((DATA_DIR / "data").glob("*.parquet"))
dataset       = ds.dataset([str(f) for f in parquet_files], format="parquet")

for batch in dataset.scanner(
        columns=["conversations","lang","task","split","dataset"],
        batch_size=256).to_batches():
    for row in batch.to_pylist():
        scanned += 1
        spec = stratum_idx.get((row.get("task"), row.get("lang")))
        if spec is None:
            continue
        msgs = parse_conversations(row)
        if msgs is None:
            skipped_parse += 1
            continue
        ids = tokenise(msgs, MAX_LENGTH)
        if len(ids) <= sliding_window:
            skipped_short += 1
            continue

        name = spec["name"]
        eligible[name] += 1
        entry = dict(stratum=name, task=spec["task"], lang=spec["lang"],
                     source_idx=scanned-1, messages=msgs, input_ids=ids,
                     seq_len=len(ids), split=row.get("split"), dataset=row.get("dataset"))

        if SAMPLING_MODE == "first":
            if len(reservoirs[name]) < TARGET_PER_STRATUM:
                reservoirs[name].append(entry)
        else:
            n = eligible[name]
            if len(reservoirs[name]) < TARGET_PER_STRATUM:
                reservoirs[name].append(entry)
            elif rng.randrange(n) < TARGET_PER_STRATUM:
                reservoirs[name][rng.randrange(TARGET_PER_STRATUM)] = entry

    if SAMPLING_MODE == "first" and all(
            len(reservoirs[s["name"]]) >= TARGET_PER_STRATUM for s in STRATA):
        break

selected = []
for sid, spec in enumerate(STRATA):
    for p in sorted(reservoirs[spec["name"]], key=lambda x: x["source_idx"]):
        p["sample_id"] = len(selected)
        selected.append(p)

print(f"scanned {scanned:,}  skipped_parse={skipped_parse}  skipped_short={skipped_short}")
print(f"selected {len(selected)} prompts:\n")
print(f"  {'stratum':<45}  {'selected':>8}  {'eligible':>8}  seq_len")
for spec in STRATA:
    rows = [p for p in selected if p["stratum"] == spec["name"]]
    elig = eligible[spec["name"]]
    lens = [p["seq_len"] for p in rows]
    print(f"  {spec['name']:<45}  {len(rows):>8}  {elig:>8}  {lens}")


scanned 320,260  skipped_parse=0  skipped_short=20445
selected 5 prompts:

  stratum                                        selected  eligible  seq_len
  chat/en                                               1    152212  [640]
  named_entity_recognition/en                           1       299  [640]
  named_entity_recognition/es                           1       301  [517]
  machine_translation/en-de                             1        29  [627]
  machine_translation_evaluation/zh_en                  1         1  [576]


In [5]:
# save prompt index
index_path = OUT_DIR / "prompt_index.jsonl"
with open(index_path, "w") as f:
    for p in selected:
        f.write(json.dumps(dict(
            sample_id=p["sample_id"], stratum=p["stratum"], task=p["task"],
            lang=p["lang"], source_idx=p["source_idx"], seq_len=p["seq_len"],
            split=p.get("split"), dataset=p.get("dataset"),
        )) + "\n")
print(f"saved {index_path}")


saved outputs/smoke/prompt_index.jsonl


## 3. The measurement

### score_pre — Gram matrix trick

For each key k we want `‖Σ_h a[q,k,h] · P[h,k,:]‖²` where `P[h,k,:] = V[h,k] @ W_o[h]ᵀ`.

Naively that's one vector per (q,k) pair — too slow. Instead we build a **Gram matrix**
`G[k,h,j] = P[h,k,:] · P[j,k,:]` and score all K keys at once:

```
score_pre² = einsum("qkh, khj, qkj → qk",  A, G, A)
```

One matrix multiply covers all late queries simultaneously.

### score_exact — leave-one-out in normed space

```python
base    = RMSNorm(pre_out)
variant = RMSNorm(pre_out − c_k)
score_exact = ‖base − variant‖₂
```

We apply this only at **anchor queries** (first late q, midpoint, last) to keep runtime tractable.


In [6]:
def score_pre_gram(attn_h, values_rep, W_o, late_q, n_heads, head_dim, hidden):
    """
    attn_h     : (H, S, S)   — [head, query, key], float32
    values_rep : (H, S, D)   — repeated-KV values, float32
    W_o        : (hidden, H·D)
    Returns (sp, P) where sp is (n_late, S) and P is (H, S, hidden).
    """
    W = W_o.float().view(hidden, n_heads, head_dim).permute(1, 0, 2)  # (H, hidden, D)
    P = torch.einsum("hkd,hod->hko", values_rep, W)                   # (H, K, hidden)
    G = torch.einsum("hko,jko->khj", P, P)                            # (K, H, H)  Gram
    A = attn_h.permute(1, 2, 0)[late_q]                               # (n_late, K, H)
    sq = torch.einsum("qkh,khj,qkj->qk", A, G, A)
    return sq.clamp_min(0).sqrt(), P


def score_exact(layernorm, pre_out, contrib):
    """L2 shift in post-RMSNorm space when `contrib` is removed from `pre_out`."""
    x = pre_out.unsqueeze(0).unsqueeze(0)
    base    = layernorm(x).squeeze().float()
    variant = layernorm(x - contrib.unsqueeze(0).unsqueeze(0)).squeeze().float()
    return float((base - variant).norm())


def build_masks(vis_idx, q, S, device):
    vis  = torch.zeros(S, dtype=torch.bool, device=device)
    vis[vis_idx] = True
    bos  = vis.clone(); bos[1:] = False
    edge = torch.zeros(S, dtype=torch.bool, device=device)
    edge[vis_idx[:min(4, len(vis_idx))]] = True
    self_ = torch.zeros(S, dtype=torch.bool, device=device); self_[q] = True
    rec  = torch.zeros(S, dtype=torch.bool, device=device)
    rec[vis_idx[max(0, len(vis_idx)-32):]] = True
    return dict(
        bos=bos, edge=edge, edge_partition=edge & ~bos,
        self=self_, recent=rec, recent_nonself=rec & ~self_,
        middle=vis & ~bos & ~(edge & ~bos) & ~rec & ~self_,
    )

def primary_group(pos, masks):
    for g in ("bos", "edge_partition", "self", "recent_nonself"):
        if masks[g][pos]: return g.replace("_partition","").replace("_nonself","")
    return "middle"


## 4. Forward hook

We monkey-patch each attention module to capture the attention weights and value
states **after** `eager_attention_forward` runs. The hook reproduces Gemma 4's
exact attention path: RoPE, QK norms, KV-sharing for layers 15–34, sliding-window mask.


In [7]:
def install_hooks(model, captured, layer_types):
    """
    Patches all 35 attention modules. Returns a `restore()` callable.
    Each forward appends one dict to `captured` with everything needed
    for score_pre and score_exact.
    """
    originals = []
    for layer_idx, layer in enumerate(model.model.language_model.layers):
        attn = layer.self_attn
        lt   = layer_types[layer_idx]
        orig = attn.forward

        def _make(idx, ltype, parent, module):
            def fwd(self, hidden_states, position_embeddings, attention_mask,
                    shared_kv_states, past_key_values=None, **kw):
                cos, sin = position_embeddings
                shape = (*hidden_states.shape[:-1], -1, self.head_dim)

                q_st = self.q_proj(hidden_states).view(shape)
                q_st = self.q_norm(q_st)
                q_st = modeling_gemma4.apply_rotary_pos_emb(q_st, cos, sin, unsqueeze_dim=2)
                q_st = q_st.transpose(1, 2)

                if self.is_kv_shared_layer:
                    k_st, v_st = shared_kv_states[self.kv_shared_layer_index]
                    k_st, v_st = k_st.to(q_st.device), v_st.to(q_st.device)
                else:
                    k_st = self.k_proj(hidden_states).view(shape)
                    v_st = (self.v_proj(hidden_states).view(shape)
                            if self.v_proj is not None else k_st)
                    k_st = self.k_norm(k_st)
                    k_st = modeling_gemma4.apply_rotary_pos_emb(k_st, cos, sin, unsqueeze_dim=2)
                    k_st, v_st = k_st.transpose(1, 2), self.v_norm(v_st).transpose(1, 2)

                if past_key_values is not None and not self.is_kv_shared_layer:
                    k_st, v_st = past_key_values.update(k_st, v_st, self.layer_idx)
                if self.store_full_length_kv:
                    shared_kv_states[self.layer_idx] = k_st, v_st

                heads, attn_w = modeling_gemma4.eager_attention_forward(
                    self, q_st, k_st, v_st, attention_mask,
                    dropout=0.0, scaling=self.scaling,
                    sliding_window=self.sliding_window, **kw)

                attn_out = self.o_proj(
                    heads.reshape(*hidden_states.shape[:-1], -1).contiguous())

                captured.append(dict(
                    layer_idx=idx, layer_type=ltype,
                    attn_weights=attn_w,          # (1, H, S, S)
                    value_states=v_st,            # (1, H_kv, S, D)
                    attn_output=attn_out,         # (1, S, hidden) — post o_proj
                    W_o=self.o_proj.weight,
                    n_kv_groups=self.num_key_value_groups,
                    head_dim=self.head_dim,
                    n_heads=int(self.config.num_attention_heads),
                    hidden=self.o_proj.weight.shape[0],
                    post_attn_ln=parent.post_attention_layernorm,
                ))
                return attn_out, attn_w
            return types.MethodType(fwd, module)

        attn.forward = _make(layer_idx, lt, layer, attn)
        originals.append((attn, orig))

    def restore():
        for mod, orig in originals:
            mod.forward = orig
    return restore


## 5. Run

For each prompt we do one forward pass and then process the captured activations:

1. Compute `score_pre` for **all late queries** via the Gram matrix (fast, vectorised).
2. Aggregate group sums → `agg_rows` (one row per late query per layer).
3. At **anchor queries** (q=512, midpoint, last), compute per-position `score_exact`
   for the top-k positions → `case_rows`.


In [8]:
agg_rows  = []   # (sample, layer, query) × group score_pre sums
case_rows = []   # (sample, layer, anchor_query, key_pos) × both scores

val_recon_error    = None
val_bos_visible    = []
val_negative_exact = 0

captured = []
restore  = install_hooks(model, captured, layer_types)

try:
    for prompt in selected:
        captured.clear()
        t0  = time.time()
        ids = torch.tensor([prompt["input_ids"]], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            model(input_ids=ids)

        S        = prompt["seq_len"]
        late_q   = list(range(sliding_window, S))
        anchor_q = {sliding_window, (sliding_window + S - 1) // 2, S - 1}
        pieces   = tokenizer.convert_ids_to_tokens(prompt["input_ids"])
        decoded  = [tokenizer.decode([t]) for t in prompt["input_ids"]]
        sid      = prompt["sample_id"]

        print(f"[{sid:02d}] {prompt['stratum']}  S={S}  "
              f"{len(late_q)} late queries  {len(anchor_q)} anchors  "
              f"fwd={time.time()-t0:.1f}s")

        for cap in captured:
            attn_h  = cap["attn_weights"][0].float()     # (H, S, S)
            values  = modeling_gemma4.repeat_kv(
                          cap["value_states"], cap["n_kv_groups"])[0].float()   # (H, S, D)
            pre_out = cap["attn_output"][0].float()       # (S, hidden)
            ln      = cap["post_attn_ln"]
            li      = cap["layer_idx"]
            lt      = cap["layer_type"]

            # ── score_pre for all late queries ────────────────────────
            sp, P = score_pre_gram(
                attn_h, values, cap["W_o"], late_q,
                cap["n_heads"], cap["head_dim"], cap["hidden"])
            # sp: (n_late, S)  P: (H, S, hidden)

            # ── one-time reconstruction check ─────────────────────────
            if val_recon_error is None:
                q0    = late_q[0]
                recon = torch.einsum("hk,hko->ko", attn_h[:, q0, :], P).sum(0)
                err   = float((recon - pre_out[q0]).norm() / pre_out[q0].norm())
                val_recon_error = err
                print(f"       reconstruction error: {err:.5f}  {'✓' if err < 0.01 else '✗'}")

            for qi, q in enumerate(late_q):
                row_a = attn_h[:, q, :]      # (H, S)
                vis   = (row_a.sum(0) > 0)
                vis_i = vis.nonzero(as_tuple=False).flatten().tolist()
                if not vis_i:
                    continue

                if lt == "sliding_attention":
                    val_bos_visible.append(bool(vis[0].item()))

                masks  = build_masks(vis_i, q, S, attn_h.device)
                sp_row = sp[qi]   # (S,)

                # group score_pre sums → agg_rows
                agg_rows.append(dict(
                    sample_id=sid, stratum=prompt["stratum"],
                    layer=li, layer_type=lt,
                    bos_pre    =float(sp_row[masks["bos"]].sum()),
                    edge_pre   =float(sp_row[masks["edge_partition"]].sum()),
                    self_pre   =float(sp_row[masks["self"]].sum()),
                    recent_pre =float(sp_row[masks["recent_nonself"]].sum()),
                    middle_pre =float(sp_row[masks["middle"]].sum()),
                    bos_attn   =float(row_a[:, masks["bos"]].sum()),
                    edge_attn  =float(row_a[:, masks["edge_partition"]].sum()),
                    recent_attn=float(row_a[:, masks["recent_nonself"]].sum()),
                    middle_attn=float(row_a[:, masks["middle"]].sum()),
                ))

                if q not in anchor_q:
                    continue

                # anchor query: exact scores for top-k + always BOS + self
                vis_sp   = sp_row[vis_i]
                top_k    = vis_sp.topk(min(ANCHOR_K, len(vis_i))).indices.tolist()
                cands    = {vis_i[i] for i in top_k} | {vis_i[0], q}
                if lt == "full_attention" and 0 in vis_i:
                    cands.add(0)

                for kp in sorted(cands):
                    cv  = torch.einsum("h,ho->o", attn_h[:, q, kp], P[:, kp, :])
                    ex  = score_exact(ln, pre_out[q], cv)
                    if ex < -1e-8:
                        val_negative_exact += 1
                    case_rows.append(dict(
                        sample_id=sid, stratum=prompt["stratum"],
                        layer=li, layer_type=lt,
                        query_pos=q, key_pos=kp,
                        position_group=primary_group(kp, masks),
                        token_piece=pieces[kp],
                        decoded_token=decoded[kp].replace("\n","\\n"),
                        attention_mass_mean=float(attn_h[:, q, kp].mean()),
                        score_pre=float(sp_row[kp]),
                        score_resid_exact=ex,
                    ))

finally:
    restore()

print(f"\nagg_rows: {len(agg_rows):,}   case_rows: {len(case_rows):,}")
bos_rate = sum(val_bos_visible) / max(len(val_bos_visible), 1)
print(f"sliding BOS visible rate  : {bos_rate:.5f}   {'✓' if bos_rate < 0.001 else '✗'}")
print(f"negative exact scores     : {val_negative_exact}   {'✓' if val_negative_exact == 0 else '✗'}")


[00] chat/en  S=640  128 late queries  3 anchors  fwd=60.2s


/var/folders/9_/b8s3b2qx4l7f5m01hyh6kcq00000gn/T/ipykernel_67542/1861438618.py:49: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  err   = float((recon - pre_out[q0]).norm() / pre_out[q0].norm())


       reconstruction error: 0.00228  ✓
[01] named_entity_recognition/en  S=640  128 late queries  3 anchors  fwd=65.6s
[02] named_entity_recognition/es  S=517  5 late queries  3 anchors  fwd=67.1s
[03] machine_translation/en-de  S=627  115 late queries  3 anchors  fwd=74.0s
[04] machine_translation_evaluation/zh_en  S=576  64 late queries  3 anchors  fwd=74.0s

agg_rows: 15,400   case_rows: 4,891
sliding BOS visible rate  : 0.00000   ✓
negative exact scores     : 0   ✓


## 6. Aggregate

In [9]:
import pandas as pd

agg   = pd.DataFrame(agg_rows)
cases = pd.DataFrame(case_rows)

# mean score_pre share by layer type
pre_cols = ["bos_pre","edge_pre","self_pre","recent_pre","middle_pre"]
totals   = agg[pre_cols].sum(axis=1).clip(lower=1e-9)
agg_share = agg[pre_cols].div(totals, axis=0)
agg["layer_scope"] = agg["layer_type"].map(
    lambda t: "sliding" if "sliding" in t else "full")
agg_share["layer_scope"] = agg["layer_scope"]

print("── score_pre share by layer type ──")
print(agg_share.groupby("layer_scope")[pre_cols].mean().round(4).to_string())
print()
print("── mean exact score by (layer_type, position_group) ──")
print(cases.groupby(["layer_type","position_group"])[["score_pre","score_resid_exact"]]
          .mean().round(4).to_string())


── score_pre share by layer type ──
             bos_pre  edge_pre  self_pre  recent_pre  middle_pre
layer_scope                                                     
full          0.1034    0.0974    0.0370      0.1554      0.6068
sliding       0.0000    0.0018    0.0494      0.3637      0.5851

── mean exact score by (layer_type, position_group) ──
                                  score_pre  score_resid_exact
layer_type        position_group                              
full_attention    bos                5.5830             8.9092
                  edge               3.6871             7.1190
                  middle             4.0434            14.8257
                  recent             3.7216            29.8873
                  self               2.1226            15.6481
sliding_attention edge               0.0550             0.0534
                  middle             4.3908             3.2223
                  recent             4.5893             9.2563
                  

In [10]:
# Cancedda check
print("── Cancedda check (full layers, BOS positions) ──")
bos_full = cases[(cases["layer_type"]=="full_attention") & (cases["position_group"]=="bos")]
if len(bos_full):
    eb = bos_full["score_resid_exact"].mean()
    ab = bos_full["attention_mass_mean"].mean()
    print(f"  mean exact_score : {eb:.4f}")
    print(f"  mean attn_mass   : {ab:.4f}")
    print(f"  ratio            : {eb/max(ab,1e-9):.2f}×")
    msg = ("BOS delivers more residual impact per unit attention than average — "
           "falsifies the low-V no-op claim ✓"
           if eb > ab else "BOS behaves like a no-op on this sample")
    print(f"  → {msg}")
else:
    print("  no BOS case rows found (need full-attention layers with visible BOS)")


── Cancedda check (full layers, BOS positions) ──
  mean exact_score : 8.9092
  mean attn_mass   : 0.0571
  ratio            : 156.11×
  → BOS delivers more residual impact per unit attention than average — falsifies the low-V no-op claim ✓


## 7. Save outputs

In [11]:
# cases.parquet
cases_path = OUT_DIR / "cases.parquet"
pq.write_table(pa.Table.from_pandas(cases, preserve_index=False), cases_path)
print(f"saved {cases_path}  ({len(cases)} rows)")


saved outputs/smoke/cases.parquet  (4891 rows)


In [12]:
# summary.json
by_type = (agg_share.groupby("layer_scope")[pre_cols].mean()
           .rename(columns=lambda c: c.replace("_pre",""))
           .to_dict(orient="index"))

summary = {
    "generated_at" : datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "model_id"     : MODEL_ID,
    "device"       : DEVICE,
    "max_length"   : MAX_LENGTH,
    "target_per_stratum" : TARGET_PER_STRATUM,
    "sampling_mode": SAMPLING_MODE,
    "num_prompts"  : len(selected),
    "active_strata": [s["name"] for s in STRATA],
    "validation"   : {
        "reconstruction_relative_error" : val_recon_error,
        "sliding_bos_visible_rate"      : bos_rate,
        "anchor_exact_negative_count"   : val_negative_exact,
        "sliding_bos_visible_for_late_query": val_bos_visible,
    },
    "aggregates"   : {
        "layer_type": {
            scope: {
                "pre_position_partition_share_mean": {
                    g.replace("_pre",""): v for g, v in vals.items()
                },
                "attention_overlap_mass_mean": {
                    "bos":    float(agg[agg["layer_scope"]==scope]["bos_attn"].mean()),
                    "edge":   float(agg[agg["layer_scope"]==scope]["edge_attn"].mean()),
                    "recent": float(agg[agg["layer_scope"]==scope]["recent_attn"].mean()),
                    "middle": float(agg[agg["layer_scope"]==scope]["middle_attn"].mean()),
                },
                "anchor_exact_position_partition_share_mean": (
                    cases[cases["layer_scope"]==scope]
                    .groupby("position_group")["score_resid_exact"].mean()
                    .reindex(["bos","edge","self","recent_nonself","middle"], fill_value=0)
                    .to_dict()
                    if "layer_scope" in cases.columns else {}
                ),
                "anchor_exact_overlap_mean": {
                    "bos": (
                        cases[(cases["layer_type"]==(
                            "full_attention" if scope=="full" else "sliding_attention"))
                            & (cases["position_group"]=="bos")]["score_resid_exact"].mean()
                        if len(cases) else 0
                    ),
                },
                "query_count"  : int((agg["layer_scope"]==scope).sum()),
                "anchor_count" : int((cases.get("layer_scope", pd.Series(dtype=str))==scope).sum()),
            }
            for scope, vals in by_type.items()
        },
    },
}
# add layer_scope to cases if not present
if "layer_scope" not in cases.columns:
    cases["layer_scope"] = cases["layer_type"].map(
        lambda t: "sliding" if "sliding" in t else "full")
# recompute anchor exact mean properly
for scope in ("sliding","full"):
    ltype = "full_attention" if scope=="full" else "sliding_attention"
    sub   = cases[cases["layer_type"]==ltype]
    by_g  = sub.groupby("position_group")["score_resid_exact"].mean().to_dict()
    summary["aggregates"]["layer_type"][scope]["anchor_exact_position_partition_share_mean"] = by_g
    summary["aggregates"]["layer_type"][scope]["anchor_count"] = len(sub)

summary_path = OUT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"saved {summary_path}")


saved outputs/smoke/summary.json


In [13]:
# report.md
slide = summary["aggregates"]["layer_type"].get("sliding", {})
full  = summary["aggregates"]["layer_type"].get("full",    {})

bos_exact = full.get("anchor_exact_position_partition_share_mean",{}).get("bos", 0)
bos_attn  = full.get("attention_overlap_mass_mean",{}).get("bos", 0)

lines = [
    "# Attention Sinks — Results",
    f"",
    f"Generated: {summary['generated_at']} · model: `{MODEL_ID}` · device: `{DEVICE}`",
    f"Prompts: {summary['num_prompts']} · strata: {len(summary['active_strata'])}",
    "",
    "## Validation",
    "",
    "| check | value | pass |",
    "|---|---|---|",
    f"| reconstruction_relative_error | {val_recon_error:.5f} | {'✓' if val_recon_error < 0.01 else '✗'} |",
    f"| sliding_bos_visible_rate | {bos_rate:.5f} | {'✓' if bos_rate < 0.001 else '✗'} |",
    f"| anchor_exact_negative_count | {val_negative_exact} | {'✓' if val_negative_exact == 0 else '✗'} |",
    f"| exact_bos > attn_bos (full) | {bos_exact:.4f} > {bos_attn:.4f} | {'✓' if bos_exact > bos_attn else '✗'} |",
    "",
    "## Score-pre share by layer type",
    "",
]
lines += ["| group | sliding_pre | full_pre |", "|---|---|---|"]
for g in ["bos","edge","self","recent","middle"]:
    sv = slide.get("pre_position_partition_share_mean",{}).get(g, 0)
    fv = full.get("pre_position_partition_share_mean",{}).get(g, 0)
    lines.append(f"| {g} | {sv:.4f} | {fv:.4f} |")

lines += [
    "",
    "## Cancedda test (full layers — BOS)",
    "",
    f"- BOS mean exact score : **{bos_exact:.4f}**",
    f"- BOS mean attn mass   : **{bos_attn:.4f}**",
    f"- Ratio                : **{bos_exact/max(bos_attn,1e-9):.2f}×**",
    "",
    ("BOS delivers **more** residual-stream impact per unit attention mass than average — "
     "**falsifies Cancedda 2024's low-V no-op claim.**"
     if bos_exact > bos_attn else
     "BOS residual impact does not exceed attention mass share on this sample."),
]

report_path = OUT_DIR / "report.md"
report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"saved {report_path}")
print()
print("── all outputs ──")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")


saved outputs/smoke/report.md

── all outputs ──
  cases.parquet  (134,917 bytes)
  prompt_index.jsonl  (938 bytes)
  report.md  (898 bytes)
  smoke_all_cases.parquet  (190,498 bytes)
  smoke_all_index.jsonl  (1,178 bytes)
  smoke_all_report.md  (24,025 bytes)
  smoke_all_summary.json  (716,897 bytes)
  smoke_es_cases.parquet  (43,838 bytes)
  smoke_es_index.jsonl  (237 bytes)
  smoke_es_report.md  (14,329 bytes)
  smoke_es_summary.json  (152,475 bytes)
  summary.json  (162,493 bytes)
